In [1]:
# ==========================================================
# Notebook 04: Local Vector Store (FAISS / Chroma)
# ==========================================================

import faiss
import pickle
import numpy as np
import pandas as pd

In [2]:
chunks_df = pd.read_csv("data/processed/chunked_corpus.csv")

with open("data/embeddings/chunk_embeddings.pkl", "rb") as file:

    embeddings = pickle.load(file)

print(embeddings.shape)

(2865, 384)


In [3]:
embeddings = np.array(embeddings).astype("float32")

print(embeddings.dtype)

float32


In [4]:
faiss.normalize_L2(embeddings)

In [5]:
embedding_dimension = embeddings.shape[1]

print(embedding_dimension)

384


In [6]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)

In [7]:
index.add(embeddings)

print("Number of vectors:", index.ntotal)

Number of vectors: 2865


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
query = "How does semantic search work?"

query_embedding = model.encode(query)

query_embedding = np.array([query_embedding]).astype("float32")

faiss.normalize_L2(query_embedding)

In [10]:
top_k = 3

scores, indices = index.search(query_embedding, top_k)

print(scores)

print(indices)

[[0.59522164 0.5588393  0.53408766]]
[[1969 1388 2028]]


In [11]:
results = chunks_df.iloc[indices[0]].copy()

results["similarity_score"] = scores[0]

results[["chunk_id", "source", "similarity_score", "chunk_text"]]

,chunk_id,source,similarity_score,chunk_text
1969,1970,book.pdf,0.595222,sks with limited data. The\nreason is that the...
1388,1389,book.pdf,0.558839,"indexing and search,\nso by default it uses L..."
2028,2029,book.pdf,0.534088,ter run a query we can quickly look up in whic...


In [12]:
def faiss_search(query, model, index, dataframe, top_k=5):

    query_embedding = model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = dataframe.iloc[indices[0]].copy()

    results["score"] = scores[0]

    return results

In [13]:
search_results = faiss_search(
    query="Tell me about FAISS indexing",
    model=model,
    index=index,
    dataframe=chunks_df,
    top_k=3,
)

search_results[["source", "score", "chunk_text"]]

,source,score,chunk_text
2033,book.pdf,0.546456,he group (Figure 9-4).\nFigure 9-4. The struct...
1388,book.pdf,0.443484,"indexing and search,\nso by default it uses L..."
1988,book.pdf,0.420457,have a closer\nlook at how it works in a minut...


In [14]:
faiss.write_index(index, "data/embeddings/faiss_index.bin")

print("FAISS index saved.")

FAISS index saved.


In [15]:
loaded_index = faiss.read_index("data/embeddings/faiss_index.bin")

print(loaded_index.ntotal)

2865


In [16]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(name="personal_notes")

In [17]:
collection.add(
    documents=chunks_df["chunk_text"].tolist(),
    embeddings=embeddings.tolist(),
    ids=[str(i) for i in chunks_df["chunk_id"]],
)

print("Collection created.")

Collection created.


In [18]:
query_embedding = model.encode("How does semantic search work?").tolist()

results = collection.query(query_embeddings=[query_embedding], n_results=3)

results

{'ids': [['1970', '1389', '2029']],
 'embeddings': None,
 'documents': [['sks with limited data. The\nreason is that these models learn useful representations of text that encode information across many dimensions,\nsuch as sentiment, topic, text structure, and more. For this reason, the embeddings of large language models can be\nused to develop a semantic search engine, fi',
   ' indexing and search,\nso by default it uses Lucene’s practical scoring function. You can find the nitty-gritty details behind the scoring\nfunction in the Elasticsearch documentation, but in brief terms it first filters the candidate documents by applying a\nBoolean test (does the document match the q',
   'ter run a query we can quickly look up in which documents the search terms appear. This works\nwell with discrete objects such as words, but does not work with continuous objects such as vectors. Each\ndocument likely has a unique vector, and therefore the index will never match with a new vector. Ins']],